In [ ]:
!pip install requests pillow ipywidgets -q

In [ ]:
import os

# Set your Fashn.ai API key here
FASHN_API_KEY = "your_api_key_here"  # Replace with your key

# Or load from environment variable
# FASHN_API_KEY = os.getenv("FASHN_API_KEY")

BASE_URL = "https://api.fashn.ai/v1"
HEADERS  = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {FASHN_API_KEY}",
}

print("API key set!" if FASHN_API_KEY and FASHN_API_KEY != 'your_api_key_here' else "Please set your API key above.")

In [ ]:
import time
import base64
import requests
from pathlib import Path
from PIL import Image
from IPython.display import display, Image as IPImage
import ipywidgets as widgets
import io

POLL_INTERVAL = 3    # seconds between status checks
MAX_WAIT      = 120  # max seconds to wait


def encode_image(image_path: str) -> str:
    """Convert local image to base64 data URI."""
    path = Path(image_path)
    suffix = path.suffix.lower()
    mime_map = {".jpg": "image/jpeg", ".jpeg": "image/jpeg",
                ".png": "image/png", ".webp": "image/webp"}
    mime = mime_map.get(suffix, "image/jpeg")
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    return f"data:{mime};base64,{b64}"


def prepare_image(image_input: str) -> str:
    """Accept URL or local path — returns ready-to-send value."""
    if image_input.startswith("http://") or image_input.startswith("https://"):
        return image_input
    return encode_image(image_input)


def submit_tryon(person_image: str, garment_image: str, category: str = "auto") -> str:
    """Submit try-on request and return prediction ID."""
    payload = {
        "model_name": "tryon-v1.6",
        "inputs": {
            "model_image":      prepare_image(person_image),
            "garment_image":    prepare_image(garment_image),
            "category":         category,
            "moderation_level": "permissive",
        },
    }
    print(f"Submitting request  (category: {category}) ...")
    resp = requests.post(f"{BASE_URL}/run", json=payload, headers=HEADERS, timeout=30)

    if resp.status_code != 200:
        raise Exception(f"API Error {resp.status_code}: {resp.text}")

    pred_id = resp.json().get("id")
    print(f"Request accepted — Prediction ID: {pred_id}")
    return pred_id


def poll_status(prediction_id: str) -> str:
    """Poll until done — returns result image URL."""
    status_url = f"{BASE_URL}/status/{prediction_id}"
    elapsed    = 0
    print("Processing", end="", flush=True)

    while elapsed < MAX_WAIT:
        time.sleep(POLL_INTERVAL)
        elapsed += POLL_INTERVAL
        print(".", end="", flush=True)

        resp   = requests.get(status_url, headers=HEADERS, timeout=15)
        data   = resp.json()
        status = data.get("status", "")

        if status == "completed":
            print(" Done!")
            return data["output"][0]

        if status in ("failed", "canceled", "timed_out"):
            raise Exception(f"Prediction {status}: {data.get('error')}")

    raise TimeoutError(f"Timed out after {MAX_WAIT} seconds.")


def download_image(url: str, save_path: str = "tryon_result.png") -> str:
    """Download result image from CDN and save locally."""
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    with open(save_path, "wb") as f:
        f.write(resp.content)
    print(f"Saved to: {save_path}")
    return save_path


def show_images_side_by_side(person_path, garment_path, result_path):
    """Display person, garment, and result side by side."""
    def load(p):
        if p.startswith("http"):
            r = requests.get(p, timeout=30)
            return Image.open(io.BytesIO(r.content)).convert("RGB")
        return Image.open(p).convert("RGB")

    person  = load(person_path)
    garment = load(garment_path)
    result  = Image.open(result_path).convert("RGB")

    # Resize all to same height
    h = 500
    def resize_h(img):
        ratio = h / img.height
        return img.resize((int(img.width * ratio), h), Image.LANCZOS)

    person  = resize_h(person)
    garment = resize_h(garment)
    result  = resize_h(result)

    gap    = 20
    total_w = person.width + garment.width + result.width + gap * 2
    canvas  = Image.new("RGB", (total_w, h + 40), (245, 245, 245))

    canvas.paste(person,  (0, 0))
    canvas.paste(garment, (person.width + gap, 0))
    canvas.paste(result,  (person.width + garment.width + gap * 2, 0))

    canvas.save("comparison.png")
    display(IPImage("comparison.png"))


print("Helper functions loaded!")

In [ ]:
# ─────────────────────────────────────────────
# Set your image paths or URLs below
# ─────────────────────────────────────────────

PERSON_IMAGE  = "person.jpg"        # Local path or URL
GARMENT_IMAGE = "garment.jpg"       # Local path or URL
OUTPUT_PATH   = "tryon_result.png"  # Where to save result

# Garment category:
#   "auto"      → Fashn detects automatically Recommended
#   "tops"      → Shirts, jackets, hoodies
#   "bottoms"   → Pants, skirts, shorts
#   "one-piece" → Dresses, jumpsuits, full outfits
CATEGORY = "auto"

# ─────────────────────────────────────────────
# Run the pipeline
# ─────────────────────────────────────────────
try:
    prediction_id = submit_tryon(PERSON_IMAGE, GARMENT_IMAGE, CATEGORY)
    result_url    = poll_status(prediction_id)

    print(f"Result URL: {result_url}")

    download_image(result_url, OUTPUT_PATH)
    print("\nTry-on complete!")

except Exception as e:
    print(f"\nError: {e}")

In [ ]:
# Show result image only
print("📸 Result Image:")
display(IPImage(OUTPUT_PATH))

In [ ]:
# Try multiple garments on the same person

person = "person.jpg"  # Same person for all

garments = [
    {"image": "shirt.jpg",    "category": "tops",      "output": "result_shirt.png"},
    {"image": "pants.jpg",    "category": "bottoms",   "output": "result_pants.png"},
    {"image": "dress.jpg",    "category": "one-piece", "output": "result_dress.png"},
]

results = []

for i, g in enumerate(garments, 1):
    print(f"\n{'='*50}")
    print(f"  Garment {i}/{len(garments)}: {g['image']}")
    print(f"{'='*50}")
    try:
        pred_id    = submit_tryon(person, g["image"], g["category"])
        result_url = poll_status(pred_id)
        saved      = download_image(result_url, g["output"])
        results.append({"garment": g["image"], "output": saved, "status": "✅ Success"})
    except Exception as e:
        results.append({"garment": g["image"], "output": None, "status": f"❌ {e}"})

print("\n\n📋 Batch Summary:")
for r in results:
    print(f"  {r['status']}  {r['garment']}  →  {r['output']}")

In [ ]:
# Display all batch results
for r in results:
    if r["output"]:
        print(f"\n👗 {r['garment']}")
        display(IPImage(r["output"], width=400))